In [92]:
from dotenv import load_dotenv
import os

load_dotenv(r"D:\QGen\.env")

api_key = os.getenv("GOOGLE_API_KEY")

print("api_key")

api_key


In [93]:
print(load_dotenv(r"D:\QGen\.env"))

True


In [94]:
import os

os.chdir(r"D:\QGen")
print(os.getcwd())

D:\QGen


In [95]:
import os
from dotenv import load_dotenv
from google import genai

# Load variables from .env file
load_dotenv()

# Now the environment variable will be available
api_key = os.getenv("GOOGLE_API_KEY")

# Quick check to ensure the key loaded from the .env file
print("API Key found:", api_key is not None)

# Create client
client = genai.Client(api_key=api_key)

# We stop here! 
print("Client successfully initialized!")

API Key found: True
Client successfully initialized!


In [96]:
from langchain_community.document_loaders import PyPDFLoader


In [97]:
%pwd

'D:\\QGen'

In [98]:
file_path = "D:\QGen\data\SDGs_Booklet_Web_En.pdf"
loader = PyPDFLoader(file_path)
data= loader.load()
print(data[2].page_content[:300])

IN THE YEAR 2015, LEADERS FROM 193 COUNTRIES OF THE WORLD 
CAME TOGETHER TO FACE THE FUTURE.
And what they saw was daunting. Famines. Drought. Wars. Plagues. Poverty. 
Not just in some faraway place, but in their own cities and towns and villages.
They knew things didn’t have to be this way. They kn


In [99]:
print(f"no. of pages={len(data)}")

no. of pages=24


In [100]:
question_gen = ""

for pages in data:
    question_gen += pages.page_content
    

In [101]:
question_gen

'IN THE YEAR 2015, LEADERS FROM 193 COUNTRIES OF THE WORLD \nCAME TOGETHER TO FACE THE FUTURE.\nAnd what they saw was daunting. Famines. Drought. Wars. Plagues. Poverty. \nNot just in some faraway place, but in their own cities and towns and villages.\nThey knew things didn’t have to be this way. They knew we had enough \nfood to feed the world, but that it wasn’t getting shared. They knew there \nwere medicines for HIV and other diseases, but they cost a lot. They knew \nthat earthquakes and floods were inevitable, but that the high death \ntolls were not. \nThey also knew that billions of people worldwide shared their hope for a \nbetter future.\nSo leaders from these countries created a plan called the Sustainable \nDevelopment Goals (SDGs). This set of 17 goals imagines a future just 15 years \noff that would be rid of poverty and hunger, and safe from the worst effects of \nclimate change. It’s an ambitious plan. \nBut there’s ample evidence that we can succeed. In the past 15 yea

In [102]:
from langchain_text_splitters import TokenTextSplitter

splitter_ques_gen = TokenTextSplitter(
    chunk_size=10000,   # adjust depending on how big you want chunks
    chunk_overlap=200
)

chunks = splitter_ques_gen.split_documents(data)  # data = loader.load()
print(len(chunks))
print(chunks[0].page_content[:300])


21
IN THE YEAR 2015, LEADERS FROM 193 COUNTRIES OF THE WORLD 
CAME TOGETHER TO FACE THE FUTURE.
And what they saw was daunting. Famines. Drought. Wars. Plagues. Poverty. 
Not just in some faraway place, but in their own cities and towns and villages.
They knew things didn’t have to be this way. They kn


In [103]:
chunk_ques_gen = splitter_ques_gen.split_text(question_gen)

In [104]:
chunk_ques_gen

['IN THE YEAR 2015, LEADERS FROM 193 COUNTRIES OF THE WORLD \nCAME TOGETHER TO FACE THE FUTURE.\nAnd what they saw was daunting. Famines. Drought. Wars. Plagues. Poverty. \nNot just in some faraway place, but in their own cities and towns and villages.\nThey knew things didn’t have to be this way. They knew we had enough \nfood to feed the world, but that it wasn’t getting shared. They knew there \nwere medicines for HIV and other diseases, but they cost a lot. They knew \nthat earthquakes and floods were inevitable, but that the high death \ntolls were not. \nThey also knew that billions of people worldwide shared their hope for a \nbetter future.\nSo leaders from these countries created a plan called the Sustainable \nDevelopment Goals (SDGs). This set of 17 goals imagines a future just 15 years \noff that would be rid of poverty and hunger, and safe from the worst effects of \nclimate change. It’s an ambitious plan. \nBut there’s ample evidence that we can succeed. In the past 15 ye

In [105]:
len(chunk_ques_gen)

1

In [106]:
from langchain_community.docstore.document import Document

In [107]:
doc_ques_gen = [Document(page_content = t) for t in chunk_ques_gen]

In [108]:
type(doc_ques_gen)


list

In [109]:
splitter_ans_gen = TokenTextSplitter(
    chunk_size=1000,   # adjust depending on how big you want chunks
    chunk_overlap=100
)


In [110]:
doc_ans_gen = splitter_ans_gen.split_documents(
    doc_ques_gen
)

doc_ans_gen

[Document(metadata={}, page_content='IN THE YEAR 2015, LEADERS FROM 193 COUNTRIES OF THE WORLD \nCAME TOGETHER TO FACE THE FUTURE.\nAnd what they saw was daunting. Famines. Drought. Wars. Plagues. Poverty. \nNot just in some faraway place, but in their own cities and towns and villages.\nThey knew things didn’t have to be this way. They knew we had enough \nfood to feed the world, but that it wasn’t getting shared. They knew there \nwere medicines for HIV and other diseases, but they cost a lot. They knew \nthat earthquakes and floods were inevitable, but that the high death \ntolls were not. \nThey also knew that billions of people worldwide shared their hope for a \nbetter future.\nSo leaders from these countries created a plan called the Sustainable \nDevelopment Goals (SDGs). This set of 17 goals imagines a future just 15 years \noff that would be rid of poverty and hunger, and safe from the worst effects of \nclimate change. It’s an ambitious plan. \nBut there’s ample evidence tha

In [111]:
len(doc_ans_gen)

5

In [ ]:
#!pip install -U langchain-google-genai

In [113]:
from langchain_google_genai import ChatGoogleGenerativeAI

In [114]:
llm_ques_gen_pipeline = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    temperature=0.3
)

In [115]:
prompt_template = """
You are an expert at creating high-quality questions from educational materials and documentation.

Your goal is to help students, professionals, and learners thoroughly understand the given content by generating meaningful and relevant questions.

Study the following text carefully:

----------------
{text}
----------------

Instructions:
- Generate questions only from the provided text.
- Create a maximum of 10 questions. 
- Prioritize the most important concepts and key points to fit within this limit.
- Avoid duplicate or repetitive questions.
- Make the questions clear, concise, and grammatically correct.
- Include a mix of factual, conceptual, and application-based questions whenever appropriate.
- Do not provide answers.
- Number each question (e.g., 1. , 2. , 3. ).
- OUTPUT FORMAT: Output ONLY the numbered questions. Do not include any introductory text, concluding remarks, or conversational filler.

QUESTIONS:
"""

In [116]:
from langchain_core.prompts import PromptTemplate

In [117]:
PROMPT_QUESTIONS = PromptTemplate(template= prompt_template, input = ["text"])

In [118]:
refine_template = """
You are an expert at creating high-quality practice questions from educational material and documentation.

Your goal is to help students and learners prepare for exams, interviews, and assessments.

We have already generated some practice questions to a certain extent:
{existing_answer}

You now have the opportunity to refine the existing questions or add new ones using the additional context provided below.

----------------
{text}
----------------

Given the new context:
- Refine the existing questions if necessary.
- Add new questions for any important concepts that are not already covered.
- Do not remove good existing questions.
- Avoid duplicate or repetitive questions.
- Make sure no important information is missed.
- Generate questions only from the provided context.
- Do not provide answers.
- If the additional context does not introduce any new information, return the original questions unchanged.


QUESTIONS:
"""

In [119]:
REFINE_PROMPT_QUESTIONS = PromptTemplate(
    input_variables = ['existing_answer', "text"],
    template = refine_template,
)

In [120]:
from langchain_classic.chains.summarize import load_summarize_chain

In [121]:
import langchain
print(langchain.__version__)

1.3.14


In [122]:
ques_gen_chain = load_summarize_chain(llm= llm_ques_gen_pipeline,
                                      chain_type = "refine",
                                      verbose = True,
                                      question_prompt = PROMPT_QUESTIONS,
                                      refine_prompt = REFINE_PROMPT_QUESTIONS)

In [123]:
ques = ques_gen_chain.run(doc_ques_gen)
print(ques)



> Entering new RefineDocumentsChain chain...


> Entering new LLMChain chain...
Prompt after formatting:

You are an expert at creating high-quality questions from educational materials and documentation.

Your goal is to help students, professionals, and learners thoroughly understand the given content by generating meaningful and relevant questions.

Study the following text carefully:

----------------
IN THE YEAR 2015, LEADERS FROM 193 COUNTRIES OF THE WORLD 
CAME TOGETHER TO FACE THE FUTURE.
And what they saw was daunting. Famines. Drought. Wars. Plagues. Poverty. 
Not just in some faraway place, but in their own cities and towns and villages.
They knew things didn’t have to be this way. They knew we had enough 
food to feed the world, but that it wasn’t getting shared. They knew there 
were medicines for HIV and other diseases, but they cost a lot. They knew 
that earthquakes and floods were inevitable, but that the high death 
tolls were not. 
They also knew that billions of p

c:\Users\shiva\anaconda3\envs\qgen\lib\site-packages\langchain_google_genai\chat_models.py:3120: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



> Finished chain.

> Finished chain.
1. In what year did leaders from 193 countries come together to create the Sustainable Development Goals (SDGs)?
2. Which organization is highlighted as one of the leading entities working to fulfill the SDGs by the year 2030?
3. What specific milestone regarding extreme poverty did the international community achieve in the 15 years leading up to the creation of the SDGs?
4. According to the text, approximately how many people around the world still live on less than $1.25 a day?
5. What percentage has the total primary school enrolment rate reached in developing regions since 2000?
6. By what year do the SDGs project that at least one in four people are likely to be affected by recurring water shortages if current paths continue?
7. How much has the number of people with access to electricity increased between the years 1990 and 2010?
8. By what fraction does the text suggest we should cut per capita food waste globally to ensure sustainable cons

In [124]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-base-en-v1.5"
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [125]:
from langchain_community.vectorstores import FAISS

vector_store = FAISS.from_documents(
    doc_ans_gen,
    embeddings
)

In [126]:

llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    google_api_key=os.getenv("GOOGLE_API_KEY"),
    temperature=0.3,
)

In [127]:
print(ques, "\n")

1. In what year did leaders from 193 countries come together to create the Sustainable Development Goals (SDGs)?
2. Which organization is highlighted as one of the leading entities working to fulfill the SDGs by the year 2030?
3. What specific milestone regarding extreme poverty did the international community achieve in the 15 years leading up to the creation of the SDGs?
4. According to the text, approximately how many people around the world still live on less than $1.25 a day?
5. What percentage has the total primary school enrolment rate reached in developing regions since 2000?
6. By what year do the SDGs project that at least one in four people are likely to be affected by recurring water shortages if current paths continue?
7. How much has the number of people with access to electricity increased between the years 1990 and 2010?
8. By what fraction does the text suggest we should cut per capita food waste globally to ensure sustainable consumption and production patterns?
9. Ap

In [128]:
ques_list = [ques]
ques_list = ques.split("\n")
ques_list


['1. In what year did leaders from 193 countries come together to create the Sustainable Development Goals (SDGs)?',
 '2. Which organization is highlighted as one of the leading entities working to fulfill the SDGs by the year 2030?',
 '3. What specific milestone regarding extreme poverty did the international community achieve in the 15 years leading up to the creation of the SDGs?',
 '4. According to the text, approximately how many people around the world still live on less than $1.25 a day?',
 '5. What percentage has the total primary school enrolment rate reached in developing regions since 2000?',
 '6. By what year do the SDGs project that at least one in four people are likely to be affected by recurring water shortages if current paths continue?',
 '7. How much has the number of people with access to electricity increased between the years 1990 and 2010?',
 '8. By what fraction does the text suggest we should cut per capita food waste globally to ensure sustainable consumption 

In [129]:
from langchain_classic.chains import RetrievalQA

In [130]:
answer_generation_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=vector_store.as_retriever()
)

In [131]:
import time

for question in ques_list:
    question = question.strip()

    if not question:
        continue

    print("Question:", question)

    try:
        # Generate the answer
        answer = answer_generation_chain.run(question)
        print("Answer:", answer)
        print("----------------------------------\n")

        # Save to file (using your excellent utf-8 addition!)
        with open("answers.txt", "a", encoding="utf-8") as f:
            f.write("Question: " + question + "\n")
            f.write("Answer: " + answer + "\n")
            f.write("----------------------------------\n\n")
            
        # WAIT 4 SECONDS before asking the next question to avoid the RPM limit
        time.sleep(4)

    except Exception as e:
        # If one question fails, print the error but keep the loop running!
        print(f"Failed to answer this question. Error: {e}")
        print("----------------------------------\n")
        time.sleep(4) # Still wait before trying the next one

Question: 1. In what year did leaders from 193 countries come together to create the Sustainable Development Goals (SDGs)?


c:\Users\shiva\anaconda3\envs\qgen\lib\site-packages\langchain_google_genai\chat_models.py:3120: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Answer: Leaders from 193 countries came together in the year **2015** to create the Sustainable Development Goals (SDGs).
----------------------------------

Question: 2. Which organization is highlighted as one of the leading entities working to fulfill the SDGs by the year 2030?


c:\Users\shiva\anaconda3\envs\qgen\lib\site-packages\langchain_google_genai\chat_models.py:3120: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Answer: Based on the provided text, the organization highlighted as one of the leading entities working to fulfill the Sustainable Development Goals (SDGs) by the year 2030 is the **United Nations Development Programme (UNDP)**.
----------------------------------

Question: 3. What specific milestone regarding extreme poverty did the international community achieve in the 15 years leading up to the creation of the SDGs?


c:\Users\shiva\anaconda3\envs\qgen\lib\site-packages\langchain_google_genai\chat_models.py:3120: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


... (11 outputs left out)

c:\Users\shiva\anaconda3\envs\qgen\lib\site-packages\langchain_google_genai\chat_models.py:3120: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Answer: Based on the provided text, plant life makes up 80% of the human diet.
----------------------------------

Question: 10. What are four suggested ways individuals can help support and reach the Sustainable Development Goals?


c:\Users\shiva\anaconda3\envs\qgen\lib\site-packages\langchain_google_genai\chat_models.py:3120: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Answer: Based on the provided text, four suggested ways individuals can help support and reach the Sustainable Development Goals are:

1. **Go shopping** (visit shop.undp.org for SDG merchandise)
2. **Make a donation** (visit undp.org/takeaction)
3. **Start a fundraiser** (to raise money, create awareness, and inspire others)
4. **Spread the word** (search for @UNDP on Twitter, Facebook, and Instagram to share content)
----------------------------------



In [ ]:
"""for question in ques_list:
    question = question.strip()

    if not question:
        continue

    print("Question:", question)

    answer = answer_generation_chain.run(question)

    print("Answer:", answer)
    print("----------------------------------\n")

    with open("answers.txt", "a", encoding="utf-8") as f:
        f.write("Question: " + question + "\n")
        f.write("Answer: " + answer + "\n")
        f.write("----------------------------------\n\n")"""